# 🎨 ComfyUI trên Kaggle Notebooks (30 giờ GPU/tuần miễn phí)

## ⚙️ THIẾT LẬP BẮT BUỘC trước khi chạy (làm 1 lần)

1. **Xác minh số điện thoại**: kaggle.com → avatar → Settings → Phone verification
   (bắt buộc để được dùng GPU + Internet)
2. Trong trang notebook, mở **⚙️ Settings (thanh bên phải)**:
   - **Accelerator** → chọn **GPU T4 x2**
   - **Internet** → bật **ON**
   - **Persistence** → chọn **Files only** (để model KHÔNG phải tải lại mỗi phiên!)
3. Chạy lần lượt **Cell 1 → 2 → 3**, copy link ở Cell 3 dán vào tab mới

💡 So với Colab: quota rõ ràng 30h/tuần (xem còn bao nhiêu ở thanh bên phải), phiên tối đa ~9-12h, ít bị ngắt ngang hơn hẳn.

⚠️ Kaggle cấm nội dung người lớn — dùng prompt lành mạnh.

In [ ]:
# ===== CELL 1: Cài ComfyUI (~2-3 phút; nếu đã bật Persistence Files thì lần sau nhanh hơn) =====
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os

# /kaggle/working được GIỮ LẠI giữa các phiên nếu bật Persistence = Files only
COMFY = '/kaggle/working/ComfyUI'

if not os.path.exists(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFY}
else:
    print('✅ ComfyUI đã có sẵn từ phiên trước (Persistence hoạt động) — chỉ cài lại thư viện.')

os.chdir(COMFY)

# Cài thư viện NHƯNG GIỮ NGUYÊN PyTorch-CUDA có sẵn của Kaggle
# (tránh lỗi 'Torch not compiled with CUDA enabled' như từng gặp trên Colab)
!grep -viE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' requirements.txt > /tmp/req_notorch.txt
!pip install -q -r /tmp/req_notorch.txt

import torch
assert torch.cuda.is_available(), '❌ Không thấy GPU! Settings (bên phải) → Accelerator → GPU T4 x2, rồi chạy lại'
print('✅ PyTorch', torch.__version__, '- GPU:', torch.cuda.get_device_name(0))

os.makedirs(f'{COMFY}/models/checkpoints', exist_ok=True)
print('\n✅ Xong Cell 1! Model đang có:')
!ls -lh {COMFY}/models/checkpoints/ || echo '(chưa có model - Cell 2 sẽ tải)'

In [ ]:
# ===== CELL 2: Tải Model (tự bỏ qua nếu đã có từ phiên trước nhờ Persistence) =====
CKPT = '/kaggle/working/ComfyUI/models/checkpoints'

# NoobAI-XL 1.1 (6.9 GB) - model anime chính
!wget -c -O {CKPT}/noobai-XL-1.1.safetensors \
  "https://huggingface.co/Laxhar/noobai-XL-1.1/resolve/main/NoobAI-XL-v1.1.safetensors"

# (TÙY CHỌN) Animagine XL 4.0 - bỏ dấu # nếu muốn (chú ý /kaggle/working giới hạn ~20GB)
#!wget -c -O {CKPT}/animagine-xl-4.0-opt.safetensors \
#  "https://huggingface.co/cagliostrolab/animagine-xl-4.0/resolve/main/animagine-xl-4.0-opt.safetensors"

!ls -lh {CKPT}
print('\n✅ Model sẵn sàng!')

In [ ]:
# ===== CELL 3: Khởi chạy ComfyUI (chạy NỀN) + tạo link truy cập =====
import subprocess, time, socket, re, os

COMFY = '/kaggle/working/ComfyUI'

# Dọn tiến trình cũ nếu chạy lại cell
!pkill -f "python main.py" 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

# Cài cloudflared
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -c -O /tmp/cf.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i /tmp/cf.deb > /dev/null 2>&1

# 1) Chạy ComfyUI NỀN
os.chdir(COMFY)
comfy_log = open('/kaggle/working/comfyui.log', 'w')
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--enable-cors-header'],
    stdout=comfy_log, stderr=subprocess.STDOUT)
print('⏳ Đang khởi động ComfyUI (30-60 giây)...')

# 2) Đợi cổng 8188 mở
for _ in range(180):
    time.sleep(1)
    if comfy.poll() is not None:
        raise RuntimeError('❌ ComfyUI bị tắt! Xem lỗi: !tail -30 /kaggle/working/comfyui.log')
    try:
        with socket.create_connection(('127.0.0.1', 8188), timeout=1):
            break
    except OSError:
        pass
else:
    raise RuntimeError('❌ Quá 3 phút chưa mở cổng. Xem log: !tail -30 /kaggle/working/comfyui.log')
print('✅ ComfyUI đã chạy!')

# 3) Tunnel cloudflared NỀN (http2 - hợp mạng VN)
cf_log = open('/kaggle/working/cloudflared.log', 'w')
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188',
     '--http-host-header', '127.0.0.1:8188', '--protocol', 'http2'],
    stdout=cf_log, stderr=subprocess.STDOUT)

# 4) Đọc link
url = None
for _ in range(60):
    time.sleep(1)
    txt = open('/kaggle/working/cloudflared.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0)
        break
print()
print('='*60)
if url:
    print('🎨 COPY LINK NÀY, DÁN VÀO THANH ĐỊA CHỈ TAB MỚI:')
    print(url)
else:
    print('⚠️ Chưa lấy được link — chạy lại cell này')
print('='*60)
print('\n💡 Cell đã xong nhưng ComfyUI vẫn chạy nền. Ảnh lưu ở /kaggle/working/ComfyUI/output')

In [ ]:
# ===== CELL 4: KIỂM TRA sức khỏe (chạy bất cứ lúc nào) =====
!curl -s -o /dev/null -w "A) ComfyUI noi bo:  HTTP %{http_code} (200 = OK)\n" --max-time 20 http://127.0.0.1:8188/system_stats

import re
txt = open('/kaggle/working/cloudflared.log').read()
m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
if m:
    url = m.group(0)
    print('   Link hien tai:', url)
    !curl -s -o /dev/null -w "B) Qua tunnel:      HTTP %{http_code} (200 = OK, 502 = ComfyUI chet, 000 = tunnel chet)\n" --max-time 40 {url}/system_stats
else:
    print('B) Khong tim thay link')

print('\n----- LOG ComfyUI (30 dòng cuối) -----')
!tail -30 /kaggle/working/comfyui.log

In [ ]:
# ===== CELL 5: Nén ảnh đã tạo để tải về (chạy trước khi tắt phiên) =====
!zip -r -q /kaggle/working/anh_da_tao.zip /kaggle/working/ComfyUI/output
print('✅ Đã nén! Tải file anh_da_tao.zip ở thanh bên phải: Output → anh_da_tao.zip')

## 📝 Ghi chú Kaggle

**Khởi động phiên mới:** bấm nút session mới → Cell 1 (nhanh nếu Persistence bật) → Cell 2 (tự bỏ qua nếu model còn) → Cell 3.

**Workflow tiếng Việt:** kéo thả các file `workflow_*_tiengviet.json` vào ComfyUI như trên Colab (dùng chung được, không cần sửa).

**Quota:** xem số giờ GPU còn lại ở thanh bên phải notebook (30h/tuần, reset thứ Bảy).

**Ảnh tạo ra:** nằm ở `/kaggle/working/ComfyUI/output` — được giữ lại nhờ Persistence; hoặc chạy Cell 5 để nén tải về máy.

**Xử lý sự cố (giống Colab):**
- Link 403 → copy dán vào thanh địa chỉ tab MỚI, đừng bấm trực tiếp
- 502 → chạy Cell 4 xem log, rồi chạy lại Cell 3
- Không thấy GPU → Settings → Accelerator → GPU T4 x2
- wget lỗi mạng → Settings → Internet → ON (cần xác minh SĐT trước)